In [1]:
from google.colab import drive
drive.mount("/content/drive")

import importlib.util, subprocess, sys
if importlib.util.find_spec("torch_geometric") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])

import ast, json, os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/content/drive/MyDrive/Threat-detection/gnn_edge_data"
GRAPH_DIR = os.path.join(DATA_DIR, "prepared_graphs")
MODEL_DIR = os.path.join(DATA_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(DATA_DIR, "train_edges_5min_enriched.parquet")
VAL_FILE = os.path.join(DATA_DIR, "validation_edges_5min_enriched.parquet")
TEST_FILE = os.path.join(DATA_DIR, "test_edges_5min_enriched.parquet")

train_graphs = torch.load(os.path.join(GRAPH_DIR, "train_graphs.pt"), map_location="cpu", weights_only=False)
val_graphs = torch.load(os.path.join(GRAPH_DIR, "validation_graphs.pt"), map_location="cpu", weights_only=False)
test_graphs = torch.load(os.path.join(GRAPH_DIR, "test_graphs.pt"), map_location="cpu", weights_only=False)

def load_labels(path):
    return (pd.read_parquet(path, columns=["window_start", "tactic_targets"])
            .sort_values("window_start", kind="quicksort").reset_index(drop=True))

train_labels = load_labels(TRAIN_FILE)
val_labels = load_labels(VAL_FILE)
test_labels = load_labels(TEST_FILE)
print("Device:", device)


Mounted at /content/drive
Device: cuda


In [2]:
def parse_labels(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set, np.ndarray)):
        values = list(value)
    elif pd.isna(value):
        return []
    else:
        text = str(value).strip()
        try:
            values = list(ast.literal_eval(text)) if text.startswith("[") else [text]
        except (ValueError, SyntaxError):
            values = [text]
    blocked = {"", "none", "unknown", "nan", "null"}
    return sorted({str(x).strip() for x in values if str(x).strip().lower() not in blocked})

classes = sorted({label for value in train_labels["tactic_targets"] for label in parse_labels(value)})
class_to_id = {label: i for i, label in enumerate(classes)}

def row_ids(graph):
    ids = graph.target_row_ids if hasattr(graph, "target_row_ids") else graph.target_row_index
    return ids.cpu().numpy()

def attach_targets(graphs, labels):
    output = []
    for graph in graphs:
        keep, targets = [], []
        for edge_id, row_id in enumerate(row_ids(graph)):
            names = [x for x in parse_labels(labels.iloc[int(row_id)]["tactic_targets"]) if x in class_to_id]
            if int(graph.y[edge_id]) == 1 and names:
                keep.append(edge_id)
                targets.append(class_to_id[names[0]])
        if keep:
            keep = torch.tensor(keep, dtype=torch.long)
            item = graph.clone()
            item.target_edge_index = graph.target_edge_index[:, keep]
            item.target_edge_attr = graph.target_edge_attr[keep]
            item.y = graph.y[keep]
            item.class_y = torch.tensor(targets, dtype=torch.long)
            output.append(item)
    return output

train_data = attach_targets(train_graphs, train_labels)
val_data = attach_targets(val_graphs, val_labels)
test_data = attach_targets(test_graphs, test_labels)
print("Classes:", len(classes), classes)
print("Labeled edges:", sum(len(g.class_y) for g in train_data), sum(len(g.class_y) for g in val_data), sum(len(g.class_y) for g in test_data))


Classes: 13 ['Collection', 'Command and Control', 'Credential Access', 'Defense Evasion', 'Discovery', 'Execution', 'Exfiltration', 'Initial Access', 'Lateral Movement', 'Persistence', 'Privilege Escalation', 'Reconnaissance', 'Resource Development']
Labeled edges: 779 52913 958


In [3]:
class TacticGraphSAGE(nn.Module):
    def __init__(self, num_classes, hidden_size=64, dropout=0.2):
        super().__init__()
        self.conv1 = SAGEConv(6, hidden_size)
        self.conv2 = SAGEConv(hidden_size, hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2 + 10, hidden_size),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size, num_classes)
        )
        self.dropout = dropout

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, data.edge_index))
        src, dst = data.target_edge_index
        return self.classifier(torch.cat([x[src], x[dst], data.target_edge_attr], dim=1))

def collect(model, loader):
    model.eval(); probabilities, targets = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            probabilities.append(torch.softmax(model(batch), dim=1).cpu())
            targets.append(batch.class_y.cpu())
    return torch.cat(probabilities).numpy(), torch.cat(targets).numpy()

def scores(targets, probabilities):
    predictions = probabilities.argmax(1)
    return {
        "accuracy": float(accuracy_score(targets, predictions)),
        "macro_f1": float(f1_score(targets, predictions, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(targets, predictions, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(targets, predictions, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(targets, predictions, average="macro", zero_division=0)),
    }


In [4]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

model = TacticGraphSAGE(len(classes)).to(device)
targets = torch.cat([g.class_y for g in train_data])
counts = torch.bincount(targets, minlength=len(classes)).float()
class_weights = torch.sqrt(len(targets) / (len(classes) * counts.clamp_min(1))).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
checkpoint_path = os.path.join(MODEL_DIR, "tactic_graphsage.pt")

best_score, waiting = -1.0, 0
for epoch in range(1, 21):
    model.train(); total_loss = total_edges = 0
    for batch in train_loader:
        batch = batch.to(device); optimizer.zero_grad()
        loss = criterion(model(batch), batch.class_y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(batch.class_y); total_edges += len(batch.class_y)

    val_probability, val_target = collect(model, val_loader)
    val_score = f1_score(val_target, val_probability.argmax(1), average="macro", zero_division=0)
    print(f"Epoch {epoch:02d} | Loss {total_loss / total_edges:.4f} | Val macro F1 {val_score:.4f}")
    if val_score > best_score:
        best_score, waiting = val_score, 0
        torch.save({"model_state_dict": model.state_dict(), "classes": classes, "task_type": "multiclass"}, checkpoint_path)
    else:
        waiting += 1
    if waiting >= 5:
        print("Early stopping"); break


Epoch 01 | Loss 2.8696 | Val macro F1 0.0049
Epoch 02 | Loss 2.5493 | Val macro F1 0.0513
Epoch 03 | Loss 2.4143 | Val macro F1 0.0683
Epoch 04 | Loss 2.3214 | Val macro F1 0.0703
Epoch 05 | Loss 2.2308 | Val macro F1 0.0704
Epoch 06 | Loss 2.1864 | Val macro F1 0.0779
Epoch 07 | Loss 2.1670 | Val macro F1 0.0766
Epoch 08 | Loss 2.0976 | Val macro F1 0.0766
Epoch 09 | Loss 2.0547 | Val macro F1 0.0778
Epoch 10 | Loss 2.0520 | Val macro F1 0.0774
Epoch 11 | Loss 2.0131 | Val macro F1 0.1495
Epoch 12 | Loss 1.8607 | Val macro F1 0.0719
Epoch 13 | Loss 1.9421 | Val macro F1 0.0735
Epoch 14 | Loss 1.8595 | Val macro F1 0.0653
Epoch 15 | Loss 1.9044 | Val macro F1 0.0612
Epoch 16 | Loss 1.8365 | Val macro F1 0.0646
Early stopping


In [5]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
val_probability, val_target = collect(model, val_loader)
test_probability, test_target = collect(model, test_loader)
validation_metrics = scores(val_target, val_probability)
test_metrics = scores(test_target, test_probability)

checkpoint.update({"validation_metrics": validation_metrics, "test_metrics": test_metrics})
torch.save(checkpoint, checkpoint_path)
with open(os.path.join(MODEL_DIR, "tactic_metrics.json"), "w") as file:
    json.dump({"validation": validation_metrics, "test": test_metrics}, file, indent=2)

print("Validation:", validation_metrics)
print("Test:", test_metrics)
print("Saved:", checkpoint_path)


Validation: {'accuracy': 0.48541946213595905, 'macro_f1': 0.1495211180887713, 'weighted_f1': 0.46695528775326484, 'macro_precision': 0.18372442166396954, 'macro_recall': 0.24530083843735412}
Test: {'accuracy': 0.6356993736951984, 'macro_f1': 0.11222773007316714, 'weighted_f1': 0.6651239820659591, 'macro_precision': 0.17748917748917747, 'macro_recall': 0.1053573654980181}
Saved: /content/drive/MyDrive/Threat-detection/gnn_edge_data/models/tactic_graphsage.pt
